# gemma-4-E4B QLoRA fine-tune — LinkedIn job agent

Runs on Colab's free T4 (16GB) or Kaggle's dual T4 (15GB each, only one used — see cell 2) because the local box driving this repo has a 6GB card — too small for gemma-4-E4B QLoRA (Unsloth's own docs put that at ~12–16GB). This notebook trains a LoRA adapter, sanity-checks it against held-out examples, then merges + quantizes to a `.gguf` you download and drop into LM Studio in place of the base `google/gemma-4-e4b` model currently configured in `orchestrator/config.json`.

**Data**: `finetune/prepare_data.py` in this repo builds `train.messages.jsonl` / `val.messages.jsonl` from every `annotation_suite`‑reviewed corpus under `data/sft/`. As of the run that produced this notebook: **411 human‑reviewed examples** (370 train / 41 val) with `reward >= 1`, out of ~6800 raw harvested steps — most harvested steps are never opened in `annotation_suite serve`, so they carry no signal and are correctly excluded. **22 of the 411 have a real screenshot embedded** (the rest either weren't screenshot-bearing steps, or the annotator explicitly stripped the image during review — `prepare_data.py` respects that and never re-attaches a removed screenshot).

**This training round is text-only.** Unsloth's vision data collator currently can't handle a dataset with any text-only rows in it (see cell 7's markdown for the tracking issue), so the 22 screenshot-bearing rows are dropped before training — cell 8 filters them out and prints the resulting counts. The base model is still loaded through `FastVisionModel` (not `FastLanguageModel`) so the vision tower survives the merge/export step intact; it's just untouched by this round's gradients. Treat this as an **initial, format‑adherence pass** — teaching the model to reliably emit the `<thinking>...</thinking>` + `<action kind="...">...</action>` shape and to imitate the dominant action kinds (`browse.open`, `browse.click`, `tool.Read`, `browse.find`) — not a multimodal pass. Revisit vision fine-tuning once Unsloth's collator handles mixed/text-only batches, or once the screenshot count has grown enough to justify a dedicated pass.

**Before running**: `python finetune/prepare_data.py "data/sft/**/*.steps.jsonl"` locally, then upload the two files it writes (`finetune/data/train.messages.jsonl`, `finetune/data/val.messages.jsonl`) when Cell 3 asks. Screenshots are embedded inline as base64 `image_url` blocks in those files already — cell 8 drops those rows for this round, nothing else to do.

**If a cell errors on an API mismatch**: this model (`gemma-4`) is very recent and Unsloth's API for it may have moved since this notebook was written (Sep 2026, from https://unsloth.ai/docs/models/gemma-4/train and https://unsloth.ai/docs/basics/vision-fine-tuning). Check those pages against the cells below — the fix is almost always a renamed argument or chat-template string, not a change in approach.

## 1. Install

In [ ]:
%%capture
import os

!pip install pip3-autoremove
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

## 2. Load base model (4-bit)

`gemma-4-E4B` is natively multimodal, loaded through `FastVisionModel` — which is what we want now that real screenshots are in the corpus, not just a text-only fallback.

In [ ]:
import os

IS_KAGGLE = os.path.exists("/kaggle/input")  # reused in cells 6 and 18 (upload/download) to pick Kaggle vs. Colab I/O

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Kaggle gives 2 GPUs; FastVisionModel's multi-GPU
    # device_map splitting isn't reliable for 4-bit vision models (unlike plain FastLanguageModel),
    # so hide the second GPU entirely rather than fight Trainer's device-parallel detection.
    # IMPORTANT: this only works in a FRESH kernel. If CUDA was already initialized in this
    # session (e.g. you ran this notebook before setting this line), the env var is a no-op —
    # restart the Kaggle session (Run > Restart Session, not just re-running cells) first.

from unsloth import FastVisionModel
import torch

assert torch.cuda.device_count() == 1, (
    f"torch still sees {torch.cuda.device_count()} GPUs — CUDA_VISIBLE_DEVICES didn't take "
    "effect because CUDA was already initialized in this kernel. Restart the Kaggle session "
    "(not just re-run cells) and run from this cell again in a clean kernel."
)

BASE_MODEL = "unsloth/gemma-4-E4B-it"  # swap here if this repo id has moved
MAX_LENGTH = 2048  # covers the corpus: p90 ~1.3k tokens, max ~2.4k tokens by rough char count (text turns; images add their own fixed token cost on top)

model, processor = FastVisionModel.from_pretrained(
    BASE_MODEL,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
    device_map={"": 0},  # single, explicit device — now a no-op safety net since
                          # CUDA_VISIBLE_DEVICES already hides GPU 1
)

## 3. Upload the exported corpus

Both from `finetune/data/` (generated by `finetune/prepare_data.py`; not in git -- `data/` is gitignored in this repo since it can carry resume/CTC/message content, and now also embedded screenshots).

**On Kaggle**: package `train.messages.jsonl` and `val.messages.jsonl` into a private Kaggle Dataset and attach it to this notebook (Notebook > Add Input). The cell below finds them anywhere under `/kaggle/input/` and copies them next to the notebook.

**On Colab**: the cell below opens an upload dialog instead -- just pick both files when prompted.

In [ ]:
if IS_KAGGLE:
    import glob, shutil

    train_matches = glob.glob("/kaggle/input/**/train.messages.jsonl", recursive=True)
    val_matches = glob.glob("/kaggle/input/**/val.messages.jsonl", recursive=True)
    assert train_matches and val_matches, (
        "train.messages.jsonl / val.messages.jsonl not found under /kaggle/input -- add a "
        "Kaggle Dataset containing both files (Notebook > Add Input), built from this "
        "repo's finetune/data/ locally, then re-run this cell."
    )
    shutil.copy(train_matches[0], "train.messages.jsonl")
    shutil.copy(val_matches[0], "val.messages.jsonl")
    print("Found and copied:")
    print(" ", train_matches[0])
    print(" ", val_matches[0])
else:
    from google.colab import files

    print("Select train.messages.jsonl and val.messages.jsonl (multi-select in the file dialog):")
    uploaded = files.upload()
    assert "train.messages.jsonl" in uploaded, "missing train.messages.jsonl"
    assert "val.messages.jsonl" in uploaded, "missing val.messages.jsonl"

## 4. Format for training

**This round trains text-only.** Unsloth's `UnslothVisionDataCollator` can't handle a dataset that mixes (or entirely lacks) image-bearing rows yet — see [unslothai/unsloth#1436](https://github.com/unslothai/unsloth/issues/1436), which hits the same failure training a VLM on text-only data. We still load the base model through `FastVisionModel` (cell 2) so the vision tower stays intact for the merge/export step later — only the *data pipeline* below changes, dropping the 22 screenshot-bearing rows and routing everything through TRL's plain-text path (`dataset_text_field="text"`, no vision collator) instead of the vision-aware one.

Revisit once Unsloth's vision collator handles this case (or the screenshot count grows enough to be worth a dedicated multimodal pass).

In [ ]:
import json
from datasets import Dataset

def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows_all = load_jsonl("train.messages.jsonl")
val_rows_all = load_jsonl("val.messages.jsonl")

def has_image(row):
    return isinstance(row["messages"][1]["content"], list)

train_rows = [r for r in train_rows_all if not has_image(r)]
val_rows = [r for r in val_rows_all if not has_image(r)]
n_dropped = (len(train_rows_all) - len(train_rows)) + (len(val_rows_all) - len(val_rows))
print(
    f"train: {len(train_rows)}  val: {len(val_rows)}  "
    f"(dropped {n_dropped} screenshot-bearing rows this round — see cell 7 markdown)"
)

def render_text(row):
    return processor.tokenizer.apply_chat_template(row["messages"], tokenize=False)

train_dataset = Dataset.from_dict({"text": [render_text(r) for r in train_rows]})
val_dataset = Dataset.from_dict({"text": [render_text(r) for r in val_rows]})
train_dataset[0]

## 5. LoRA adapter

`finetune_vision_layers=False`: 22 image examples is nowhere near enough to safely adapt the vision tower itself — that risks overfitting or degrading its general screenshot understanding for a handful of memorized examples. Leaving it frozen still lets images flow through as features into the LoRA-tuned language/attention/mlp layers, so the model gets *some* signal on "here's a screenshot, here's the right action" without retraining how it sees. Revisit this once there are meaningfully more annotated screenshots.

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=3407,
)

## 6. Train

370 → (fewer after dropping screenshot rows this round, see cell 8's printed count) train examples, batch 1 x grad-accum 8 = effective batch 8. 3 epochs is a starting point for a corpus this size — watch `eval_loss`; if it's still dropping at the end, this is worth a longer run once there's more annotated data, and if it's *rising* before epoch 3, cut epochs back (this little data overfits fast).

Plain TRL text pipeline (`dataset_text_field="text"`) — no vision collator needed since every row here is text-only this round (see cell 7).

In [ ]:
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=processor.tokenizer,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        optim="adamw_8bit",
        weight_decay=0.01,
        max_grad_norm=0.3,
        bf16=True,
        logging_steps=5,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        seed=3407,
        output_dir="outputs",
        report_to="none",
        max_length=MAX_LENGTH,
    ),
)

trainer_stats = trainer.train()

## 7. Sanity-check before exporting

Run a few held-out val examples through the fine-tuned model and eyeball the `<action>` output against the human-approved ground truth. Do this before the (slow) GGUF export below — no point exporting a multi-GB file if the model is clearly off.

In [ ]:
FastVisionModel.for_inference(model)

import random

sample_rows = random.sample(val_rows, min(5, len(val_rows)))

for row in sample_rows:
    prompt_messages = row["messages"][:2]  # system + user, drop the ground-truth assistant turn
    ground_truth = row["messages"][2]["content"]
    inputs = processor.tokenizer.apply_chat_template(
        prompt_messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to(model.device)
    out = model.generate(**inputs, max_new_tokens=256, use_cache=True)
    predicted = processor.tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print("=" * 80)
    print("GROUND TRUTH:\n", ground_truth[:500])
    print("-" * 80)
    print("PREDICTED:\n", predicted[:500])

## 8. Save adapter + export merged GGUF

`save_pretrained_gguf` merges the LoRA adapter into the base weights and quantizes in one step. `q4_k_m` matches the quantization level typical for local LM Studio use on limited VRAM — same ballpark as the GGUF you're already running locally.

In [ ]:
ADAPTER_DIR = "gemma4-e4b-linkedin-lora"
GGUF_DIR = "gemma4-e4b-linkedin-merged"

model.save_pretrained(ADAPTER_DIR)
processor.tokenizer.save_pretrained(ADAPTER_DIR)

model.save_pretrained_gguf(
    GGUF_DIR,
    processor.tokenizer,
    quantization_method="q4_k_m",
)

## 9. Download

The `.gguf` is likely a few GB.

**On Kaggle**: there's no in-session download widget -- the cell below copies it into `/kaggle/working/gguf_export/`. Commit the notebook (Save Version > Save & Run All) and download the file from that version's **Output** tab once the run finishes.

**On Colab**: the cell below calls `files.download()` directly. If it stalls or the session disconnects, mount Drive instead (`from google.colab import drive; drive.mount('/content/drive')`) and `!cp` the file there, then download from Drive's own UI.

In [ ]:
import glob

gguf_files = glob.glob(f"{GGUF_DIR}/*.gguf")
print("Found:", gguf_files)

if IS_KAGGLE:
    import shutil

    out_dir = "/kaggle/working/gguf_export"
    os.makedirs(out_dir, exist_ok=True)
    for f in gguf_files:
        dest = os.path.join(out_dir, os.path.basename(f))
        shutil.copy(f, dest)
        print(f"Copied to {dest}")
    print(
        "Commit this notebook (Save Version > Save & Run All) and grab the file(s) from "
        "that version's 'Output' tab once it finishes -- Kaggle has no in-session "
        "browser download like Colab's files.download()."
    )
else:
    from google.colab import files

    for f in gguf_files:
        files.download(f)

## 10. Bring it back into LM Studio

1. Move the downloaded `.gguf` into LM Studio's models folder, in its own subfolder — e.g. `<LM Studio models dir>/local/gemma-4-E4B-linkedin/gemma-4-E4B-linkedin.Q4_K_M.gguf` (LM Studio expects `<publisher>/<model-name>/<file>.gguf`).
2. In LM Studio, reload the models list and load that model on the local server (same port your `orchestrator/config.json` already points `base_url` at). Confirm LM Studio's own model card still shows vision support after the merge — it should, since only LoRA weights on the language side changed, but worth a glance.
3. Update `orchestrator/config.json`'s `"model"` field to whatever name LM Studio shows for it (check the server's `/v1/models` endpoint or the LM Studio UI — it won't necessarily be `gemma-4-E4B-linkedin`, LM Studio derives the served name from the gguf metadata/folder).
4. Run `python -m orchestrator run "List the available skills."` from the repo root and confirm it responds sanely before trusting it with a real cycle. If you want to specifically check the vision path, `python -m orchestrator run --vision "Take a screenshot of the current page and describe what you see."` (needs a connected browser driver — see the root `AGENTS.md`).